# Relative elevation & depression — Copernicus DEM GLO-30 (multi-city, `FLOODS_SITE`)

Diagnostic layers for flood NBS site screening (`models/nbs_flood_mechanism_type/` / `transformation/nbs_screening/docs/recommended-datasets.md` — Decision area 1).

**Source:** [Copernicus DEM GLO-30](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_DEM_GLO30) (`COPERNICUS/DEM/GLO30`) — 30 m DSM, EGM2008 vertical datum.

| Output | Description |
|--------|-------------|
| `sites/<site_slug>/data/input/<prefix>_dem_glo30_30m.tif` | Elevation (m) exported from GEE |
| `sites/<site_slug>/data/output/<prefix>_relative_elevation_30m.tif` | **Low-lying index** 0–1 (1 = lowest elevation in POA) |
| `sites/<site_slug>/data/output/<prefix>_depression_mask_30m.tif` | **Sink mask** 0/1 (D8 local depression, no outlet) |
| `sites/<site_slug>/data/output/<prefix>_depression_depth_30m.tif` | **Depression depth** (m) = filled DEM − original DEM |

**Definitions (distinct from slope):**
- *Relative elevation* — how low a pixel is **relative to the rest of POA** (screening proxy for low-lying accumulation).
- *Depression* — topographic **sink** where water can pond (D8 + priority-flood fill depth).
- *Slope* — steepness in degrees; computed separately in `landslides/script/slope_from_dem.ipynb`.

**Workflow:** GEE export DEM → local Python (rasterio + numpy) → GeoTIFF outputs.


In [ ]:
# Site configuration — transformation/flood_hazard city configs
import os
import sys
from pathlib import Path

_HERE = Path.cwd().resolve()
_FLOOD_HAZARD = None
for _candidate in [_HERE, *_HERE.parents]:
    _probe = _candidate / "flood_hazard" if _candidate.name != "flood_hazard" else _candidate
    if (_probe / "site_config.py").is_file() and (_probe / "config" / "sites").is_dir():
        _FLOOD_HAZARD = _probe
        break
if _FLOOD_HAZARD is None:
    raise FileNotFoundError("Could not locate transformation/flood_hazard from notebook cwd")

sys.path.insert(0, str(_FLOOD_HAZARD))
from site_config import load_site_config

FLOOD_HAZARD_ROOT = _FLOOD_HAZARD
SITE_SLUG = os.environ.get("FLOODS_SITE", "porto_alegre")
SITE_CONFIG = load_site_config(SITE_SLUG, FLOOD_HAZARD_ROOT)
SITE_ROOT = SITE_CONFIG["paths_abs"]["site_root"]
INPUT_DIR = SITE_CONFIG["paths_abs"]["data_input"]
INTERMEDIATE_DIR = SITE_CONFIG["paths_abs"]["data_intermediate"]
OUTPUT_DIR = SITE_CONFIG["paths_abs"]["data_output"]
OUT_ROOT = SITE_CONFIG["paths_abs"]["out"]
CACHE_DIR = SITE_CONFIG["paths_abs"]["cache"]
STYLES_DIR = SITE_CONFIG["paths_abs"]["styles"]
OUTPUT_PREFIX = SITE_CONFIG["output_prefix"]
LAYER_FILES = SITE_CONFIG["layers"]

for _p in (INPUT_DIR, INTERMEDIATE_DIR, OUTPUT_DIR, OUT_ROOT, CACHE_DIR, STYLES_DIR):
    Path(_p).mkdir(parents=True, exist_ok=True)

print(f"Flood site: {SITE_CONFIG['display_name']} ({SITE_SLUG})")
print(f"Config: {SITE_CONFIG['config_path']}")
print(f"DEM input -> {INPUT_DIR / LAYER_FILES['dem']}")
print(f"Diagnostics -> {OUTPUT_DIR}")


## Step 1 — Export Copernicus DEM from GEE (30 m)

Per the [GEE catalog](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_DEM_GLO30), mosaic the image collection and `.reproject()` before terrain analysis.


In [ ]:
import json
import ee


ee.Initialize(project='eecc-maureen')  # adjust Earth Engine project if needed


def load_site_roi() -> ee.Geometry:
    boundary_path = SITE_CONFIG["boundary_path_abs"]
    if boundary_path.exists():
        data = json.loads(boundary_path.read_text())
        if data.get("type") == "FeatureCollection":
            features = [
                ee.Feature(ee.Geometry(feature["geometry"]), feature.get("properties", {}))
                for feature in data.get("features", [])
                if feature.get("geometry")
            ]
            if features:
                return ee.FeatureCollection(features).geometry()
        if data.get("type") == "Feature":
            return ee.Geometry(data["geometry"])
        if data.get("type") in {"Polygon", "MultiPolygon", "GeometryCollection"}:
            return ee.Geometry(data)

    return ee.Geometry.Rectangle(SITE_CONFIG["bbox"])


roi = load_site_roi()
site_geom = roi


dem = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .select('DEM')
    .filterBounds(roi)
    .mosaic()
    .reproject(crs='EPSG:4326', scale=30)
    .clip(roi)
    .toFloat()
)

stats = dem.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), '', True),
    geometry=roi,
    scale=250,
    maxPixels=1e9,
).getInfo()
print('DEM stats (m, sampled at 250 m):', stats)

task = ee.batch.Export.image.toDrive(
    image=dem,
    description=SITE_CONFIG['layers']['dem'].removesuffix('.tif'),
    folder='gee_exports',
    fileNamePrefix=SITE_CONFIG['layers']['dem'].removesuffix('.tif'),
    region=roi,
    scale=30,
    crs='EPSG:4326',
    maxPixels=1e13,
    fileFormat='GeoTIFF',
)
task.start()
print('GEE export started:', task.id)
print(f"When complete, download {SITE_CONFIG['layers']['dem']} -> sites/{SITE_SLUG}/data/input/")


## Step 2 — Download DEM

When the task finishes in the [GEE Tasks console](https://code.earthengine.google.com/tasks), download `<prefix>_dem_glo30_30m.tif` from the `gee_exports` Google Drive folder and save to:

``transformation/flood_hazard/sites/<city>/data/input/<prefix>_dem_glo30_30m.tif``

If you already have the DEM (or the catalog COG), set `DEM_PATH` in Step 3 accordingly.


## Step 3 — Compute relative elevation & depression (local)

**Relative elevation (`low_lying_pct` proxy):** percentile rank of elevation inverted within valid POA pixels — `1.0` = lowest cell, `0.0` = highest.

**Depression mask:** D8 flow direction with no downslope neighbour and no lower neighbour (sink cells).

**Depression depth:** priority-flood filled DEM minus original elevation (metres).


In [ ]:
from __future__ import annotations

import heapq
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEM_PATH = INPUT_DIR / SITE_CONFIG['layers']['dem']
OUT_REL_ELEV = OUTPUT_DIR / LAYER_FILES["relative_elevation"]
OUT_DEP_MASK = OUTPUT_DIR / LAYER_FILES["depression_mask"]
OUT_DEP_DEPTH = OUTPUT_DIR / LAYER_FILES["depression_depth"]

# D8 neighbour offsets (row, col) — ESRI-style direction index 0–7
D8_DR = np.array([-1, -1, 0, 1, 1, 1, 0, -1], dtype=np.int32)
D8_DC = np.array([0, 1, 1, 1, 0, -1, -1, -1], dtype=np.int32)
SQ2 = np.sqrt(2.0)
D8_DIST = np.array([1, SQ2, 1, SQ2, 1, SQ2, 1, SQ2], dtype=np.float64)


def valid_dem_mask(dem: np.ndarray, nodata) -> np.ndarray:
    mask = np.isfinite(dem)
    if nodata is not None and not (isinstance(nodata, float) and np.isnan(nodata)):
        mask &= dem != nodata
    mask &= dem > -500  # exclude void / ocean placeholders
    return mask


def relative_elevation_index(dem: np.ndarray, valid: np.ndarray) -> np.ndarray:
    """0–1 index: 1 = lowest elevation among valid cells (POA low-lying proxy)."""
    out = np.full(dem.shape, np.nan, dtype=np.float32)
    vals = dem[valid].astype(np.float64)
    if vals.size == 0:
        return out
    order = np.argsort(vals, kind='mergesort')
    ranks = np.empty(vals.size, dtype=np.float64)
    ranks[order] = np.arange(vals.size)
    denom = max(vals.size - 1, 1)
    out[valid] = (1.0 - ranks / denom).astype(np.float32)
    return out


def d8_flow_direction(dem: np.ndarray, valid: np.ndarray) -> np.ndarray:
    """Return int8 array: 0–7 = D8 direction, -1 = flat/pit/no outlet."""
    h, w = dem.shape
    flow = np.full((h, w), -1, dtype=np.int8)
    for r in range(h):
        for c in range(w):
            if not valid[r, c]:
                continue
            z = dem[r, c]
            best_drop = 0.0
            best_dir = -1
            for d in range(8):
                nr, nc = r + D8_DR[d], c + D8_DC[d]
                if nr < 0 or nr >= h or nc < 0 or nc >= w or not valid[nr, nc]:
                    continue
                drop = (z - dem[nr, nc]) / D8_DIST[d]
                if drop > best_drop:
                    best_drop = drop
                    best_dir = d
            flow[r, c] = best_dir
    return flow


def depression_sink_mask(dem: np.ndarray, valid: np.ndarray, flow: np.ndarray) -> np.ndarray:
    """Binary mask: D8 pit with no lower neighbour (screening-scale sinks)."""
    h, w = dem.shape
    sinks = np.zeros((h, w), dtype=np.uint8)
    for r in range(h):
        for c in range(w):
            if not valid[r, c] or flow[r, c] != -1:
                continue
            z = dem[r, c]
            is_lowest = True
            for d in range(8):
                nr, nc = r + D8_DR[d], c + D8_DC[d]
                if nr < 0 or nr >= h or nc < 0 or nc >= w or not valid[nr, nc]:
                    continue
                if dem[nr, nc] < z:
                    is_lowest = False
                    break
            if is_lowest:
                sinks[r, c] = 1
    return sinks


def priority_flood_fill(dem: np.ndarray, valid: np.ndarray) -> np.ndarray:
    """Priority-flood depression fill; inf/nodata treated as barriers."""
    h, w = dem.shape
    filled = np.where(valid, dem.astype(np.float64), np.inf)
    visited = np.zeros((h, w), dtype=bool)
    heap: list[tuple[float, int, int]] = []

    for r in range(h):
        for c in range(w):
            if not valid[r, c]:
                continue
            on_edge = r == 0 or c == 0 or r == h - 1 or c == w - 1
            if on_edge:
                heapq.heappush(heap, (filled[r, c], r, c))
                visited[r, c] = True

    while heap:
        z, r, c = heapq.heappop(heap)
        if z > filled[r, c]:
            filled[r, c] = z
        for d in range(8):
            nr, nc = r + D8_DR[d], c + D8_DC[d]
            if nr < 0 or nr >= h or nc < 0 or nc >= w or not valid[nr, nc]:
                continue
            if visited[nr, nc]:
                continue
            visited[nr, nc] = True
            elev = max(filled[nr, nc], z)
            filled[nr, nc] = elev
            heapq.heappush(heap, (elev, nr, nc))

    return filled


def write_geotiff(path: Path, arr: np.ndarray, profile: dict, nodata=None) -> None:
    out = profile.copy()
    out.update(dtype=rasterio.float32 if arr.dtype != np.uint8 else rasterio.uint8, count=1, nodata=nodata, compress='deflate')
    with rasterio.open(path, 'w', **out) as dst:
        dst.write(arr.astype(out['dtype']), 1)


assert DEM_PATH.exists(), f'Missing DEM: {DEM_PATH} — run Step 1–2 first'

with rasterio.open(DEM_PATH) as src:
    dem = src.read(1).astype(np.float32)
    profile = src.profile
    nodata = src.nodata

valid = valid_dem_mask(dem, nodata)
print(f'DEM shape: {dem.shape}, valid cells: {valid.sum():,} / {valid.size:,}')
print(f'Elevation range (valid): {dem[valid].min():.1f} – {dem[valid].max():.1f} m')

rel_elev = relative_elevation_index(dem, valid)
flow = d8_flow_direction(dem, valid)
dep_mask = depression_sink_mask(dem, valid, flow)
filled = priority_flood_fill(dem, valid)
dep_depth = np.where(valid, np.maximum(filled - dem.astype(np.float64), 0.0), np.nan).astype(np.float32)

write_geotiff(OUT_REL_ELEV, rel_elev, profile, nodata=np.nan)
write_geotiff(OUT_DEP_MASK, dep_mask, profile, nodata=255)
write_geotiff(OUT_DEP_DEPTH, dep_depth, profile, nodata=np.nan)

print('Wrote:')
for p in (OUT_REL_ELEV, OUT_DEP_MASK, OUT_DEP_DEPTH):
    print(' ', p.resolve())


## Step 4 — Inspect outputs


In [ ]:
def pct(arr, valid_mask, q):
    v = arr[valid_mask & np.isfinite(arr)]
    return float(np.percentile(v, q)) if v.size else float('nan')

print('Relative elevation (0=high, 1=low):')
print(f'  P10={pct(rel_elev, valid, 10):.3f}  P50={pct(rel_elev, valid, 50):.3f}  P90={pct(rel_elev, valid, 90):.3f}')
print(f'Depression sinks (mask=1): {dep_mask.sum():,} cells ({100 * dep_mask.sum() / valid.sum():.2f}% of valid)')
print(f'Depression depth (m): P50={pct(dep_depth, valid & (dep_depth > 0), 50):.2f}  P90={pct(dep_depth, valid & (dep_depth > 0), 90):.2f}  max={np.nanmax(dep_depth):.2f}')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title, cmap, vmin, vmax in [
    (axes[0], rel_elev, 'Relative elevation (1=lowest)', 'YlGnBu_r', 0, 1),
    (axes[1], dep_depth, 'Depression depth (m)', 'Blues', 0, np.nanpercentile(dep_depth[valid], 99)),
    (axes[2], dep_mask, 'Depression sink mask', 'Greys', 0, 1),
]:
    masked = np.where(valid, data, np.nan)
    im = ax.imshow(masked, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


## Notes

- **Screening only** — 30 m DSM includes buildings/vegetation; not suitable for engineering design.
- **Relative elevation** matches the NBS POC concept `low_lying_pct` but at native 30 m for POA.
- **Depression mask** uses D8 pits (same logic as `scripts/fetch-elevation-data.ts` in NBS Project Preparation).
- **Depression depth** uses priority-flood fill; large flat areas may show zero depth until a true sink exists.
- Next step for catalog: register outputs in `geospatial-data/catalog/datasets.yaml` and publish COG + value tiles.


## Step 5 — Convert Outputs to COG and Generate Tiles

Publish the local terrain-derived flood screening rasters for web maps: COG + colorized XYZ tiles + value-encoded XYZ tiles for hover lookup.

Inputs:
- `sites/<site_slug>/data/output/<prefix>_relative_elevation_30m.tif`
- `sites/<site_slug>/data/output/<prefix>_depression_mask_30m.tif`
- `sites/<site_slug>/data/output/<prefix>_depression_depth_30m.tif`

Requires GDAL CLI (`gdal_translate`, `gdaldem`, `gdal_calc.py`, `gdal2tiles.py`) and matching color tables in `data/`.


In [ ]:
# Convert terrain-derived flood screening GeoTIFFs to COG + visual tiles + value tiles.
from pathlib import Path
import shutil
import subprocess
import numpy as np
import rasterio


def resolve_gdal2tiles_python() -> str:
    gdal2tiles = shutil.which("gdal2tiles.py")
    if not gdal2tiles:
        raise RuntimeError("gdal2tiles.py not found in PATH")
    return gdal2tiles


def publish_layer(layer_key: str, colors_name: str, categorical: bool = False) -> None:
    in_tif = OUTPUT_DIR / LAYER_FILES[layer_key]
    slug = Path(LAYER_FILES[layer_key]).stem
    out_dir = OUT_ROOT / slug
    colors_txt = STYLES_DIR / colors_name
    out_dir.mkdir(parents=True, exist_ok=True)

    cog_tif = out_dir / f"{slug}_cog.tif"
    colorized_tif = out_dir / f"{slug}_colorized.tif"
    value_encoded_tif = out_dir / f"{slug}_value_encoded_rgb.tif"
    tiles_dir = out_dir / "tiles_visual"
    value_tiles_dir = out_dir / "tiles_values"
    decode_txt = out_dir / f"{slug}_value_tiles_decode.txt"

    print(f"Input: {in_tif}")
    print(f"Output dir: {out_dir}")
    assert in_tif.exists(), f"Missing: {in_tif}"
    assert colors_txt.exists(), f"Missing colors: {colors_txt}"

    subprocess.run([
        "gdal_translate", str(in_tif), str(cog_tif),
        "-of", "COG", "-co", "COMPRESS=DEFLATE", "-co", "PREDICTOR=2",
    ], check=True)
    print(f"Created COG: {cog_tif}")

    subprocess.run([
        "gdaldem", "color-relief", str(in_tif), str(colors_txt), str(colorized_tif),
        "-alpha", "-co", "COMPRESS=LZW",
    ], check=True)
    print(f"Created colorized raster: {colorized_tif}")

    if tiles_dir.exists():
        shutil.rmtree(tiles_dir)
    subprocess.run([
        resolve_gdal2tiles_python(), "--tiledriver=PNG", "--webviewer=none",
        "--zoom=8-15", "--resampling=near", "--xyz",
        str(colorized_tif), str(tiles_dir),
    ], check=True)
    print(f"Visual tiles: {tiles_dir}")

    with rasterio.open(in_tif) as src:
        arr = src.read(1).astype(np.float32)
        profile = src.profile.copy()

    if categorical:
        encoded = np.clip(np.nan_to_num(arr, nan=0.0), 0, 65535).astype(np.uint16)
        decode = "encoded_int = R + 256*G\nvalue = encoded_int\n"
    else:
        encoded = np.clip(np.rint(np.nan_to_num(arr, nan=0.0) * 10000), 0, 65535).astype(np.uint16)
        decode = "encoded_int = R + 256*G\nvalue = encoded_int / 10000.0\n"

    r = (encoded & 0xFF).astype(np.uint8)
    g = ((encoded >> 8) & 0xFF).astype(np.uint8)
    b = np.zeros_like(r)
    profile.update(count=3, dtype="uint8", nodata=None, compress="lzw")
    with rasterio.open(value_encoded_tif, "w", **profile) as dst:
        dst.write(r, 1); dst.write(g, 2); dst.write(b, 3)

    if value_tiles_dir.exists():
        shutil.rmtree(value_tiles_dir)
    subprocess.run([
        resolve_gdal2tiles_python(), "--tiledriver=PNG", "--webviewer=none",
        "--zoom=8-15", "--resampling=near", "--xyz",
        str(value_encoded_tif), str(value_tiles_dir),
    ], check=True)
    decode_txt.write_text(decode)
    print(f"Value tiles: {value_tiles_dir}")
    print(f"Decode metadata: {decode_txt}")


publish_layer("relative_elevation", "relative_elevation_30m_colors.txt", categorical=False)
publish_layer("depression_mask", "depression_mask_30m_colors.txt", categorical=True)
publish_layer("depression_depth", "depression_depth_30m_colors.txt", categorical=False)
print("Publish complete for", SITE_CONFIG["display_name"])
